In [0]:
%sql
SELECT * 
FROM dlt_hc_fraud_detection.bronze_silver_gold.bronze_claims
WHERE claim_id IN (1000,1001);

In [0]:
%sql
SELECT claim_id, COUNT(*) 
FROM dlt_hc_fraud_detection.bronze_silver_gold.silver_claims
WHERE claim_id IN (1000,1001)
GROUP BY claim_id;

In [0]:
%sql
SELECT COUNT(*) AS bronze_claim_count
FROM dlt_hc_fraud_detection.bronze_silver_gold.bronze_claims;

In [0]:
%sql
SELECT *
FROM dlt_hc_fraud_detection.bronze_silver_gold.bronze_claims
WHERE claim_id IS NULL;

In [0]:
%sql
DESCRIBE TABLE dlt_hc_fraud_detection.bronze_silver_gold.bronze_payments;

In [0]:
%sql
SELECT claim_id, COUNT(*) AS cnt
FROM dlt_hc_fraud_detection.bronze_silver_gold.bronze_claims
GROUP BY claim_id
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT claim_id, COUNT(*) AS cnt
FROM dlt_hc_fraud_detection.bronze_silver_gold.silver_claims
GROUP BY claim_id
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM dlt_hc_fraud_detection.bronze_silver_gold.bronze_claims) AS bronze_cnt,
  (SELECT COUNT(*) FROM dlt_hc_fraud_detection.bronze_silver_gold.silver_claims) AS silver_cnt;

In [0]:
%sql
WITH latest_bronze AS (
  SELECT *
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY claim_id ORDER BY ingestion_ts DESC) rn
    FROM dlt_hc_fraud_detection.bronze_silver_gold.bronze_claims
  )
  WHERE rn = 1
)
SELECT *
FROM latest_bronze lb
LEFT JOIN dlt_hc_fraud_detection.bronze_silver_gold.silver_claims s
ON lb.claim_id = s.claim_id
WHERE lb.claim_amount <> s.claim_amount;

In [0]:
%sql
SELECT fraud_flag, COUNT(*)
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
GROUP BY fraud_flag;

In [0]:
%sql
SELECT *
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE paid_amount > claim_amount;

In [0]:
%sql
SELECT
  (SELECT SUM(claim_amount)
   FROM dlt_hc_fraud_detection.bronze_silver_gold.silver_claims) AS silver_total,
  (SELECT SUM(claim_amount)
   FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals) AS gold_total;

In [0]:
%sql
SELECT
  s.provider_id,
  COUNT(*) AS detail_count,
  g.total_claims
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals s
JOIN dlt_hc_fraud_detection.bronze_silver_gold.gold_claim_summary g
ON s.provider_id = g.provider_id
GROUP BY s.provider_id, g.total_claims
HAVING COUNT(*) <> g.total_claims;

In [0]:
%sql
SELECT *
FROM dlt_hc_fraud_detection.bronze_silver_gold.bronze_claims
WHERE ingestion_ts < to_timestamp(claim_date, 'dd-MM-yyyy');